# Text-to-Image FLUX Multiple Prompts 🗒️🗒️🗒️🖼️ 🖼️ 🖼️

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

## Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg, save_gif
from IPython.display import Image as IPythonImage
import torch

## Load pipeline

Load a pipeline with Hugging Face Diffusers.

In [ ]:
# Create pipeline (2 min load on A100)
from diffusers import FluxPipeline
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16, token=hf_token)
pipe.enable_model_cpu_offload()

## Config

You can override parameters here.

In [ ]:
Config.OUTPUT_DIR = '/content/drive/MyDrive/iaac_genai/outputs'

Config.PROMPT = "front elevation of modern townhouse in {season} in Boston made from {material}, foreground with {tree_type} trees, at {time}, 4k, hd, raw photo, uhd"
Config.SEED = 7797676568
Config.STEPS = 28
Config.FPS = 1 # GIF frames per second

Config.AUTHOR = 'James McBennett'
Config.ALGO_TYPE = 'Text to Image'
Config.ALGO_NAME = 'Flux Prompt Variations'

Config.check()

## Create Prompts

In [ ]:
# Create prompt variations
import random
prompts = []

# Define your custom variants and use them above in Config.PROMPT.
# Here, we use style, season, and material, but you can change/add.

variants = {                                                        #
    'material': ["wood", "stone", "brownstone", "zinc", "copper", "concrete", "reinforced concrete", "glass", "red brick", "yellow brick", "adobe mud", "fabric"],
    'season': ["Summer", "Winter", "Spring", "Autumn"],
    'time': ["morning", "sunrise", "noon", "sundown", "dusk", "night"],
    'tree_type': ["pink cherry blossom", "oak", "maple", "willow", "palm"]
}
for material in variants['material']:
    new_prompt = Config.PROMPT.format(
        season=random.choice(variants['season']),
        time=random.choice(variants['time']),
        tree_type=random.choice(variants['tree_type']),
        material=material,
    )
    prompts.append(new_prompt)
    print(new_prompt)

In [ ]:
#optional: reduce number of prompts/images to generate

total_output_images = 3

prompts = prompts[:total_output_images]
print(f'Produced {len(prompts)} prompt variations.\n')
for prompt in prompts:
    print(f'› {prompt}')

## Generate & Save

In [ ]:
# Generate
frames = []
for i, prompt in enumerate(prompts):
    Config.PROMPT = prompt
    generator = torch.Generator(Config.TORCH_DEVICE).manual_seed(Config.SEED)
    image = pipe(Config.PROMPT, num_inference_steps=Config.STEPS, generator=generator).images[0]
    set_image_path()

    # Save image
    save_image(image)
    display(image)
    frames.append(image)
    print(f'{i}: {prompt}')

    # Save yml metadata
    save_yml(pipe)

    # Save svg parameters image
    save_svg({
      'SEED': Config.SEED,
      'STEPS': Config.STEPS,
      'Google Colab': '',
    })

##Gif

In [ ]:
gif_path = save_gif(frames)
IPythonImage(filename=gif_path)